In [1]:
import coffea
import hist
import pickle
from coffea import util
import matplotlib.pyplot as plt
import itertools
import os, sys
import mplhep as hep
hep.style.use("CMS")

sys.path.append('../python/')
from functions import getCoffeaFilenames, getHist

useOldHTcut = True
oldHTstr = ''
if useOldHTcut:
    oldHTstr = '_oldHTcut'

In [2]:
f1 = util.load('../outputs/TTbar_2016APV_700to1000'+oldHTstr+'.coffea')
f2 = util.load('../outputs/TTbar_2016APV_1000toInf'+oldHTstr+'.coffea')
# f3 = util.load('../outputs/JetHT_2016APVB.coffea')

In [3]:
ttagcats = ['at', 'pret', '2t']
btagcats = ["0b", "1b", "2b"]
ycats = ['cen', 'fwd']                                                                                                          
anacats = [ t+b+y for t,b,y in itertools.product( ttagcats, btagcats, ycats) ]
label_dict = {i: label for i, label in enumerate(anacats)}

In [4]:
f1['weights']

defaultdict_accumulator(float,
                        {'nominal': 7797746326.180325,
                         'btagDown': 8192559755.715698,
                         'prefiringDown': 7833152616.212212,
                         'prefiringUp': 7760396698.224126,
                         'pileupUp': 7757754317.652974,
                         'pileupDown': 7833414249.908769,
                         'pdfUp': 7852116224.81321,
                         'q2Down': 5666020553.114925,
                         'toptagsfDown': 7443464667.798609,
                         'toptagxsDown': 7173926620.085899,
                         'q2Up': 10751727971.77908,
                         'toptagxsUp': 8421566032.274753,
                         'btagUp': 7307284773.689591,
                         'toptagsfUp': 8817408932.913166,
                         'lumiDown': 7704173370.266162,
                         'lumiUp': 7891319282.09449,
                         'pdfDown': 7906486123.446095,
            

In [5]:
for i,j in enumerate(anacats):
    print(i, j)

0 at0bcen
1 at0bfwd
2 at1bcen
3 at1bfwd
4 at2bcen
5 at2bfwd
6 pret0bcen
7 pret0bfwd
8 pret1bcen
9 pret1bfwd
10 pret2bcen
11 pret2bfwd
12 2t0bcen
13 2t0bfwd
14 2t1bcen
15 2t1bfwd
16 2t2bcen
17 2t2bfwd


In [ ]:
lumi = {
    "2016APV": 19800.,
    "2016": 16120., #35920 - 19800
    "2016all": 35920,
    "2017": 41530./10.,
    "2018": 59800./10., #59740./10., #Blinding
    "Full": 46053. # 137190. Blinded
}

def MakeCMSLabel(AX, CatInt, IOV, isData=True):
    dytext = ''
    if 'cen' in anacats[CatInt]:
        dytext = r'$\Delta y$ < 1.0'
    elif 'fwd' in anacats[CatInt]:
        dytext = r'$\Delta y$ > 1.0'

    btext = ''
    if '0b' in anacats[CatInt]:
        btext = '0 b-tags'
    elif '1b' in anacats[CatInt]:
        btext = '1 b-tag'
    elif '2b' in anacats[CatInt]:
        btext = '2 b-tags'
        
    text = f'Preliminary:    {btext}, {dytext}'
    
    hep.cms.label(text, data=False, lumi='{0:0.1f}'.format(lumi[IOV]/1000.), loc=0, fontsize=17, ax=AX)

In [ ]:
# Syst = ['prefiring', 'pileup', 'pdf', 'q2', 'toptagxs', 'btag', 'toptagsf', 'lumi', 'jes', 'jer']

Syst = 'jer'
iov = '2016all'

for icat in range(12, 18):
    fig, (ax, rax) = plt.subplots(nrows=2, height_ratios=[3, 1])
    
    if 'all' in iov:
        Nom_apv = getHist('ttbarmass', 'TTbar', False, '2016APV', sum_axes=[], integrate_axes={'anacat':icat, 'systematic':'nominal'})
        Up_apv = getHist('ttbarmass', 'TTbar', False, '2016APV', sum_axes=[], integrate_axes={'anacat':icat, 'systematic':f'{Syst}Up'})
        Down_apv = getHist('ttbarmass', 'TTbar', False, '2016APV', sum_axes=[], integrate_axes={'anacat':icat, 'systematic':f'{Syst}Down'})
        Nom_noapv = getHist('ttbarmass', 'TTbar', False, '2016', sum_axes=[], integrate_axes={'anacat':icat, 'systematic':'nominal'})
        Up_noapv = getHist('ttbarmass', 'TTbar', False, '2016', sum_axes=[], integrate_axes={'anacat':icat, 'systematic':f'{Syst}Up'})
        Down_noapv = getHist('ttbarmass', 'TTbar', False, '2016', sum_axes=[], integrate_axes={'anacat':icat, 'systematic':f'{Syst}Down'})
        
        Nom = Nom_apv + Nom_noapv
        Up = Up_apv + Up_noapv
        Down = Down_apv + Down_noapv
    
    S = hist.Stack(Up, Nom, Down)
    S.plot(ax=ax, stack=False, histtype="step", lw=3, label=['Up','Nom.','Down'], color=['blue', 'green', 'red'])
    
    ax.set_ylim(bottom=0.)
    ax.set_xlabel('')
    ax.set_xlim(800, 6000)
    ax.legend()
    
    MakeCMSLabel(ax, icat, iov, False)
    
    
    
    UpCorr =  Up / Nom.values()
    DownCorr =  Down / Nom.values()
    
    # print(Down.values())
    
    ratioUp = hep.histplot(UpCorr, ax=rax, histtype='step', color='blue')
    ratioDown = hep.histplot(DownCorr, ax=rax, histtype='step', color='red')
    
    rax.set_ylim(0.5,1.5)
    rax.axhline(1, color='green')
    rax.set_ylabel('Syst./Nom.')
    rax.set_xlim(800, 6000)
    
    if Syst == 'toptagsf': rax.set_ylim(0.5,2)
    elif Syst == 'lumi': rax.set_ylim(0.95,1.05)
    elif Syst == 'prefiring': rax.set_ylim(0.95,1.05)
    elif Syst == 'pileup': rax.set_ylim(0.80,1.20)
    elif Syst == 'pdf': rax.set_ylim(0.80,1.20)
    elif Syst == 'q2': rax.set_ylim(0.20,1.80)
    elif Syst == 'btag': rax.set_ylim(0.75,1.25)
    elif Syst == 'toptagxs': rax.set_ylim(0.80,1.20)
    elif Syst == 'jer': rax.set_ylim(0.80,1.20)
        
    
    leg = plt.text(0.73, 0.55, f'TTbar Sim.\nSignal Region\n{Syst} correction',
                    fontsize=16,
                    weight='bold',
                    transform=ax.transAxes
                   )
    plt.legend()
    plt.show()

In [ ]:
# for icat in range(12, 17):
#     fig = plt.figure(icat+1)
    
#     f1['ttbarmass_fine'][{'anacat':icat, 'systematic':'q2Up'}].project('ttbarmass').plot(label='Up')
#     f1['ttbarmass_fine'][{'anacat':icat, 'systematic':'nominal'}].project('ttbarmass').plot(label='Nom.')
#     f1['ttbarmass_fine'][{'anacat':icat, 'systematic':'q2Down'}].project('ttbarmass').plot(label='Down')
    
#     plt.title("APV " + anacats[icat])
#     plt.legend()

In [ ]:
# for icat in range(17):
#     fig = plt.figure(icat+1)
#     f1['ttbarmass_bare'].project("anacat", "ttbarmass")[icat,:].plot(density=True, label="Bare")
#     f1['ttbarmass_fine'].project("anacat", "ttbarmass")[icat,:].plot(density=True, label="Weighted")
#     plt.title("APV " + anacats[icat])
#     plt.legend()